# Matching System con razze reali e razze visive

Questo notebook integra nel sistema di matching anche le razze visive stimate dalle immagini nel notebook 08.

La logica è:

- la razza reale principale (`breed1_label`) ha il peso più alto;
- la razza reale secondaria (`breed2_label`) ha peso medio;
- la razza visiva stimata dalle immagini ha peso minore, perché non è certa;
- per la razza visiva viene considerata anche la percentuale/confidenza del modello.

Esempio:

- Labrador reale in `breed1_label` → punteggio alto;
- Labrador reale in `breed2_label` → punteggio medio;
- Labrador stimato dalle immagini al 20% → punteggio parziale maggiore;
- Labrador stimato dalle immagini all'1% → punteggio molto basso.


## 1. Import delle librerie

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

## 2. Caricamento dei dati

In [2]:
dogs = pd.read_csv("../data/processed/dogs_clean.csv")
bert_embeddings = np.load("../data/processed/bert_embeddings.npy")

profiles_path = Path("../data/family_profiles.json")

with open(profiles_path, "r", encoding="utf-8") as f:
    family_profiles = json.load(f)

visual_breeds_path = Path("../data/results/visual_breed_predictions_mixed_breed.csv")
visual_breeds = pd.read_csv(visual_breeds_path)

print("Dataset cani:", dogs.shape)
print("Embedding BERT:", bert_embeddings.shape)
print("Predizioni razze visive:", visual_breeds.shape)

visual_breeds.head()

Dataset cani: (8132, 32)
Embedding BERT: (8132, 768)
Predizioni razze visive: (5923, 12)


,PetID,Name,breed1_label,breed2_label,PhotoAmt,num_images_used,visual_breed_1,visual_breed_1_score,visual_breed_2,visual_breed_2_score,visual_breed_3,visual_breed_3_score
0,3422e4906,Brisco,Mixed Breed,NaN,7.0,7,toy terrier,0.129274,wire-haired fox terrier,0.069780,Chihuahua,0.052176
1,5842f1ff5,Miko,Mixed Breed,NaN,8.0,8,Rottweiler,0.030565,schipperke,0.024801,toy terrier,0.021206
2,850a43f90,Hunter,Mixed Breed,NaN,3.0,3,schipperke,0.096710,Labrador retriever,0.048065,flat-coated retriever,0.033992
3,97aa9eeac,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,9.0,9,French bulldog,0.028243,Brittany spaniel,0.022413,toy terrier,0.020661
4,8b693ca84,Bear,Mixed Breed,NaN,7.0,7,toy terrier,0.097166,Italian greyhound,0.034993,wire-haired fox terrier,0.023989


## 3. Merge con le razze visive

Le razze visive sono disponibili soprattutto per i cani `Mixed Breed`.  
Per gli altri cani le colonne visive rimangono vuote o con score pari a 0.


In [3]:
dogs = dogs.merge(
    visual_breeds[[
        "PetID",
        "visual_breed_1",
        "visual_breed_1_score",
        "visual_breed_2",
        "visual_breed_2_score",
        "visual_breed_3",
        "visual_breed_3_score",
        "num_images_used"
    ]],
    on="PetID",
    how="left"
)

dogs["visual_breed_1"] = dogs["visual_breed_1"].fillna("")
dogs["visual_breed_2"] = dogs["visual_breed_2"].fillna("")
dogs["visual_breed_3"] = dogs["visual_breed_3"].fillna("")

dogs["visual_breed_1_score"] = dogs["visual_breed_1_score"].fillna(0.0)
dogs["visual_breed_2_score"] = dogs["visual_breed_2_score"].fillna(0.0)
dogs["visual_breed_3_score"] = dogs["visual_breed_3_score"].fillna(0.0)
dogs["num_images_used"] = dogs["num_images_used"].fillna(0)

dogs[[
    "PetID",
    "Name",
    "breed1_label",
    "breed2_label",
    "visual_breed_1",
    "visual_breed_1_score",
    "visual_breed_2",
    "visual_breed_2_score",
    "visual_breed_3",
    "visual_breed_3_score"
]].head()

,PetID,Name,breed1_label,breed2_label,visual_breed_1,visual_breed_1_score,visual_breed_2,visual_breed_2_score,visual_breed_3,visual_breed_3_score
0,3422e4906,Brisco,Mixed Breed,NaN,toy terrier,0.129274,wire-haired fox terrier,0.069780,Chihuahua,0.052176
1,5842f1ff5,Miko,Mixed Breed,NaN,Rottweiler,0.030565,schipperke,0.024801,toy terrier,0.021206
2,850a43f90,Hunter,Mixed Breed,NaN,schipperke,0.096710,Labrador retriever,0.048065,flat-coated retriever,0.033992
3,97aa9eeac,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,French bulldog,0.028243,Brittany spaniel,0.022413,toy terrier,0.020661
4,8b693ca84,Bear,Mixed Breed,NaN,toy terrier,0.097166,Italian greyhound,0.034993,wire-haired fox terrier,0.023989


## 4. Scelta del profilo famiglia

In [4]:
selected_profile_id = "family_children_apartment"

family_profile = family_profiles[selected_profile_id]

family_profile

{'profile_name': 'Famiglia con bambini in appartamento',
 'applicant_age': '31-60',
 'children': True,
 'dog_experience': 'some',
 'housing': 'apartment',
 'garden': False,
 'preferred_gender': 'indifferent',
 'preferred_age': 'young',
 'preferred_size': 'small',
 'preferred_fur': 'short',
 'daily_time': '2-4',
 'activity_level': 'moderate',
 'preferred_breeds': ['Labrador Retriever', 'Golden Retriever'],
 'ideal_dog_description': 'We are looking for a small, affectionate and friendly dog suitable for a family with children. The dog should be calm at home, sociable, easy to manage and suitable for apartment life.'}

## 5. Funzioni di compatibilità

La funzione `compute_breed_score` considera:

1. razza reale principale;
2. razza reale secondaria;
3. razze visive stimate.

La razza visiva riceve un punteggio minore e proporzionale alla confidenza:

```text
visual_score = visual_confidence * visual_weight
```

con `visual_weight = 0.5`.

Quindi una corrispondenza visiva al 20% pesa più di una all'1%.


In [5]:
def normalize_text(text):
    return str(text).strip().lower()


def match_preference(preference, dog_value):
    if preference == "indifferent":
        return 1.0
    return 1.0 if preference == dog_value else 0.0


def size_similarity(preferred_size, dog_size):
    if preferred_size == "indifferent":
        return 1.0

    size_order = {
        "small": 0,
        "medium": 1,
        "large": 2,
        "extra_large": 3
    }

    if preferred_size not in size_order or dog_size not in size_order:
        return 0.0

    distance = abs(size_order[preferred_size] - size_order[dog_size])

    if distance == 0:
        return 1.0
    elif distance == 1:
        return 0.5
    else:
        return 0.0


def compute_breed_score(dog, family):
    preferred_breeds = family.get("preferred_breeds", [])

    if len(preferred_breeds) == 0:
        return 1.0

    preferred_breeds = [normalize_text(breed) for breed in preferred_breeds]

    breed1 = normalize_text(dog.get("breed1_label", ""))
    breed2 = normalize_text(dog.get("breed2_label", ""))

    # Razza reale principale: massimo peso
    if breed1 in preferred_breeds:
        return 1.0

    # Razza reale secondaria: peso medio
    if breed2 in preferred_breeds:
        return 0.7

    # Razze visive: peso minore e proporzionale alla confidenza
    visual_weight = 0.5

    visual_candidates = [
        (normalize_text(dog.get("visual_breed_1", "")), float(dog.get("visual_breed_1_score", 0.0))),
        (normalize_text(dog.get("visual_breed_2", "")), float(dog.get("visual_breed_2_score", 0.0))),
        (normalize_text(dog.get("visual_breed_3", "")), float(dog.get("visual_breed_3_score", 0.0)))
    ]

    visual_scores = []

    for visual_breed, visual_confidence in visual_candidates:
        if visual_breed in preferred_breeds:
            visual_scores.append(visual_confidence * visual_weight)

    if len(visual_scores) > 0:
        return max(visual_scores)

    return 0.0

## 6. Compatibilità strutturata

La razza entra nello `structured_score` con peso 3.

Questo significa che la preferenza di razza è importante, ma se la razza è solo visiva il suo contributo resta più basso.


In [6]:
def compute_structured_score(dog, family):
    if family["daily_time"] == "<1":
        return 0.0

    if dog["maturity_size_label"] == "extra_large" and not family["garden"]:
        return 0.0

    if family["dog_experience"] == "none" and dog["maturity_size_label"] == "extra_large":
        return 0.0

    weighted_scores = []

    gender_score = match_preference(family["preferred_gender"], dog["gender_label"])
    age_score = match_preference(family["preferred_age"], dog["age_group"])
    size_score = size_similarity(family["preferred_size"], dog["maturity_size_label"])
    fur_score = match_preference(family["preferred_fur"], dog["fur_length_label"])
    breed_score = compute_breed_score(dog, family)

    weighted_scores.append((gender_score, 1))
    weighted_scores.append((age_score, 1))
    weighted_scores.append((size_score, 1))
    weighted_scores.append((fur_score, 1))
    weighted_scores.append((breed_score, 3))

    if family["children"]:
        children_score = 1.0 if dog["age_group"] in ["puppy", "young"] else 0.5
        weighted_scores.append((children_score, 1))

    if dog["maturity_size_label"] == "large":
        garden_score = 1.0 if family["garden"] else 0.2
        weighted_scores.append((garden_score, 1))
    elif dog["maturity_size_label"] == "extra_large":
        weighted_scores.append((1.0, 1))
    else:
        weighted_scores.append((1.0, 1))

    if family["daily_time"] == "1-2":
        time_score = 0.6 if dog["age_group"] == "puppy" else 0.8
        weighted_scores.append((time_score, 1))
    else:
        weighted_scores.append((1.0, 1))

    if family["dog_experience"] == "none":
        if dog["maturity_size_label"] == "large":
            experience_score = 0.2
        elif dog["age_group"] == "puppy":
            experience_score = 0.5
        else:
            experience_score = 1.0
    else:
        experience_score = 1.0

    weighted_scores.append((experience_score, 1))

    if family["applicant_age"] == "over_60":
        senior_score = 1.0 if dog["age_group"] in ["adult", "senior"] else 0.4
        weighted_scores.append((senior_score, 1))

    total_score = sum(score * weight for score, weight in weighted_scores)
    total_weight = sum(weight for score, weight in weighted_scores)

    return total_score / total_weight

## 7. Calcolo degli score strutturati

In [7]:
dogs_experiment = dogs.copy()

dogs_experiment["breed_score"] = dogs_experiment.apply(
    lambda row: compute_breed_score(row, family_profile),
    axis=1
)

dogs_experiment["structured_score"] = dogs_experiment.apply(
    lambda row: compute_structured_score(row, family_profile),
    axis=1
)

dogs_experiment[[
    "Name",
    "breed1_label",
    "breed2_label",
    "visual_breed_1",
    "visual_breed_1_score",
    "visual_breed_2",
    "visual_breed_2_score",
    "visual_breed_3",
    "visual_breed_3_score",
    "breed_score",
    "structured_score"
]].head(10)

,Name,breed1_label,breed2_label,visual_breed_1,visual_breed_1_score,visual_breed_2,visual_breed_2_score,visual_breed_3,visual_breed_3_score,breed_score,structured_score
0,Brisco,Mixed Breed,NaN,toy terrier,0.129274,wire-haired fox terrier,0.069780,Chihuahua,0.052176,0.000000,0.500000
1,Miko,Mixed Breed,NaN,Rottweiler,0.030565,schipperke,0.024801,toy terrier,0.021206,0.000000,0.590909
2,Hunter,Mixed Breed,NaN,schipperke,0.096710,Labrador retriever,0.048065,flat-coated retriever,0.033992,0.024032,0.597463
3,Siu Pak & Her 6 Puppies,Mixed Breed,NaN,French bulldog,0.028243,Brittany spaniel,0.022413,toy terrier,0.020661,0.000000,0.590909
4,Bear,Mixed Breed,NaN,toy terrier,0.097166,Italian greyhound,0.034993,wire-haired fox terrier,0.023989,0.000000,0.590909
5,Peanut,Mixed Breed,NaN,Labrador retriever,0.117036,golden retriever,0.027283,standard poodle,0.013308,0.058518,0.515959
6,Lost Dog,Mixed Breed,NaN,golden retriever,0.106757,Labrador retriever,0.043042,American Staffordshire terrier,0.027622,0.053378,0.514558
7,Max,Terrier,Shih Tzu,,0.000000,,0.000000,,0.000000,0.000000,0.454545
8,Blackie,Mixed Breed,Mixed Breed,Norwegian elkhound,0.204912,German shepherd,0.018285,toy terrier,0.014740,0.000000,0.681818
9,Beauty,Mixed Breed,NaN,Labrador retriever,0.143148,flat-coated retriever,0.074805,schipperke,0.054338,0.071574,0.610429


## 8. Similarità semantica tramite BERT

In [8]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [9]:
def get_bert_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = bert_model(**inputs)

    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()


family_embedding = get_bert_embedding(family_profile["ideal_dog_description"])

semantic_scores = cosine_similarity(
    family_embedding.reshape(1, -1),
    bert_embeddings
).flatten()

dogs_experiment["bert_similarity"] = (semantic_scores + 1) / 2

dogs_experiment[["Name", "bert_similarity"]].head()

,Name,bert_similarity
0,Brisco,0.907398
1,Miko,0.876957
2,Hunter,0.925550
3,Siu Pak & Her 6 Puppies,0.822069
4,Bear,0.827598


## 9. Score finale

In [10]:
dogs_experiment["final_score"] = (
    0.7 * dogs_experiment["structured_score"] +
    0.3 * dogs_experiment["bert_similarity"]
)

dogs_experiment[[
    "Name",
    "breed_score",
    "structured_score",
    "bert_similarity",
    "final_score"
]].head()

,Name,breed_score,structured_score,bert_similarity,final_score
0,Brisco,0.000000,0.500000,0.907398,0.622220
1,Miko,0.000000,0.590909,0.876957,0.676724
2,Hunter,0.024032,0.597463,0.925550,0.695889
3,Siu Pak & Her 6 Puppies,0.000000,0.590909,0.822069,0.660257
4,Bear,0.000000,0.590909,0.827598,0.661916


## 10. Ranking finale

In [11]:
ranking = dogs_experiment.sort_values(
    by="final_score",
    ascending=False
)[[
    "Name",
    "Age",
    "breed1_label",
    "breed2_label",
    "visual_breed_1",
    "visual_breed_1_score",
    "visual_breed_2",
    "visual_breed_2_score",
    "visual_breed_3",
    "visual_breed_3_score",
    "breed_score",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "PhotoAmt",
    "structured_score",
    "bert_similarity",
    "final_score",
    "Description"
]].head(20)

ranking

,Name,Age,breed1_label,breed2_label,visual_breed_1,visual_breed_1_score,visual_breed_2,visual_breed_2_score,visual_breed_3,visual_breed_3_score,breed_score,age_group,gender_label,maturity_size_label,fur_length_label,PhotoAmt,structured_score,bert_similarity,final_score,Description
5743,Bobby,8,Golden Retriever,NaN,,0.000000,,0.000000,,0.000000,1.0,young,male,small,short,5.0,1.000000,0.884814,0.965444,Golden retriever 8months old.Need a loving hom...
8089,Pacino,12,Labrador Retriever,Mixed Breed,,0.000000,,0.000000,,0.000000,1.0,young,male,medium,short,7.0,0.954545,0.911148,0.941526,PAcino was found at a park in Damansara Kim on...
7580,"Mummy Moon, Luna And Ariel",15,Golden Retriever,Labrador Retriever,,0.000000,,0.000000,,0.000000,1.0,young,female,medium,short,3.0,0.954545,0.902593,0.938960,Three pups were rescued during Thaipusam and f...
3697,2 Black Dude,8,Labrador Retriever,Mixed Breed,,0.000000,,0.000000,,0.000000,1.0,young,mixed,medium,short,7.0,0.954545,0.901899,0.938751,"siblings brother and sister, big paws and shin..."
2549,FLUFFY,22,Golden Retriever,NaN,,0.000000,,0.000000,,0.000000,1.0,young,female,medium,short,2.0,0.954545,0.899024,0.937889,i am the owner for this dog and i am looking f...
1285,Whiskey,12,Labrador Retriever,Mixed Breed,,0.000000,,0.000000,,0.000000,1.0,young,male,medium,short,2.0,0.954545,0.897704,0.937493,Whiskey is our own pet. An offspring of my aun...
1898,Nil,12,Labrador Retriever,Mixed Breed,,0.000000,,0.000000,,0.000000,1.0,young,male,medium,short,1.0,0.954545,0.864164,0.927431,I'm a labrador retriever cross local. I have t...
3309,Snow,12,Labrador Retriever,Mixed Breed,,0.000000,,0.000000,,0.000000,1.0,young,female,small,long,4.0,0.909091,0.913736,0.910484,Snow used to have a home nearby my area but wh...
37,Baby / Bubbles,2,Labrador Retriever,Jack Russell Terrier,,0.000000,,0.000000,,0.000000,1.0,puppy,male,small,short,5.0,0.909091,0.894526,0.904721,3 month-old male puppy for adoption. a cross b...
2757,Simba,2,Labrador Retriever,Beagle,,0.000000,,0.000000,,0.000000,1.0,puppy,female,small,short,0.0,0.909091,0.838549,0.887928,Simba is very loving. She is also very playful...


## 11. Esportazione CSV

In [12]:
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

export_columns = [
    "final_score",
    "structured_score",
    "breed_score",
    "bert_similarity",
    "PetID",
    "Name",
    "breed1_label",
    "breed2_label",
    "visual_breed_1",
    "visual_breed_1_score",
    "visual_breed_2",
    "visual_breed_2_score",
    "visual_breed_3",
    "visual_breed_3_score",
    "num_images_used",
    "Age",
    "age_group",
    "gender_label",
    "maturity_size_label",
    "fur_length_label",
    "PhotoAmt",
    "Description"
]

full_ranking = dogs_experiment.sort_values(
    by="final_score",
    ascending=False
)[export_columns]

full_ranking = full_ranking.round({
    "final_score": 4,
    "structured_score": 4,
    "breed_score": 4,
    "bert_similarity": 4,
    "visual_breed_1_score": 4,
    "visual_breed_2_score": 4,
    "visual_breed_3_score": 4
})

output_file = results_dir / f"ranking_{selected_profile_id}_with_visual_breeds.csv"

full_ranking.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("File salvato in:")
print(output_file)

print("Numero cani esportati:")
print(len(full_ranking))

full_ranking.head(20)

File salvato in:
..\data\results\ranking_family_children_apartment_with_visual_breeds.csv
Numero cani esportati:
8132


,final_score,structured_score,breed_score,bert_similarity,PetID,Name,breed1_label,breed2_label,visual_breed_1,visual_breed_1_score,...,visual_breed_3,visual_breed_3_score,num_images_used,Age,age_group,gender_label,maturity_size_label,fur_length_label,PhotoAmt,Description
5743,0.9654,1.0000,1.0,0.8848,01c6d3224,Bobby,Golden Retriever,NaN,,0.0000,...,,0.0000,0.0,8,young,male,small,short,5.0,Golden retriever 8months old.Need a loving hom...
8089,0.9415,0.9545,1.0,0.9111,b3a6f353d,Pacino,Labrador Retriever,Mixed Breed,,0.0000,...,,0.0000,0.0,12,young,male,medium,short,7.0,PAcino was found at a park in Damansara Kim on...
7580,0.9390,0.9545,1.0,0.9026,295dac1ce,"Mummy Moon, Luna And Ariel",Golden Retriever,Labrador Retriever,,0.0000,...,,0.0000,0.0,15,young,female,medium,short,3.0,Three pups were rescued during Thaipusam and f...
3697,0.9388,0.9545,1.0,0.9019,ab9dd85b2,2 Black Dude,Labrador Retriever,Mixed Breed,,0.0000,...,,0.0000,0.0,8,young,mixed,medium,short,7.0,"siblings brother and sister, big paws and shin..."
2549,0.9379,0.9545,1.0,0.8990,e246ecfdc,FLUFFY,Golden Retriever,NaN,,0.0000,...,,0.0000,0.0,22,young,female,medium,short,2.0,i am the owner for this dog and i am looking f...
1285,0.9375,0.9545,1.0,0.8977,902114441,Whiskey,Labrador Retriever,Mixed Breed,,0.0000,...,,0.0000,0.0,12,young,male,medium,short,2.0,Whiskey is our own pet. An offspring of my aun...
1898,0.9274,0.9545,1.0,0.8642,2d88536fc,Nil,Labrador Retriever,Mixed Breed,,0.0000,...,,0.0000,0.0,12,young,male,medium,short,1.0,I'm a labrador retriever cross local. I have t...
3309,0.9105,0.9091,1.0,0.9137,9430d84c5,Snow,Labrador Retriever,Mixed Breed,,0.0000,...,,0.0000,0.0,12,young,female,small,long,4.0,Snow used to have a home nearby my area but wh...
37,0.9047,0.9091,1.0,0.8945,1e62e6022,Baby / Bubbles,Labrador Retriever,Jack Russell Terrier,,0.0000,...,,0.0000,0.0,2,puppy,male,small,short,5.0,3 month-old male puppy for adoption. a cross b...
2757,0.8879,0.9091,1.0,0.8385,a64d70c84,Simba,Labrador Retriever,Beagle,,0.0000,...,,0.0000,0.0,2,puppy,female,small,short,0.0,Simba is very loving. She is also very playful...


## 12. Spiegazione della raccomandazione

In [13]:
def explain_breed_match(dog, family):
    preferred_breeds = [normalize_text(breed) for breed in family.get("preferred_breeds", [])]

    if len(preferred_breeds) == 0:
        return None

    breed1 = normalize_text(dog.get("breed1_label", ""))
    breed2 = normalize_text(dog.get("breed2_label", ""))

    if breed1 in preferred_breeds:
        return "razza principale reale compatibile"

    if breed2 in preferred_breeds:
        return "seconda razza reale compatibile"

    visual_candidates = [
        (normalize_text(dog.get("visual_breed_1", "")), float(dog.get("visual_breed_1_score", 0.0))),
        (normalize_text(dog.get("visual_breed_2", "")), float(dog.get("visual_breed_2_score", 0.0))),
        (normalize_text(dog.get("visual_breed_3", "")), float(dog.get("visual_breed_3_score", 0.0)))
    ]

    best_visual_match = None

    for visual_breed, visual_score in visual_candidates:
        if visual_breed in preferred_breeds:
            if best_visual_match is None or visual_score > best_visual_match[1]:
                best_visual_match = (visual_breed, visual_score)

    if best_visual_match is not None:
        return f"razza visiva compatibile ({best_visual_match[0]}, score={best_visual_match[1]:.2f})"

    return None


def explain_recommendation(dog, family, all_dogs):
    reasons = []

    if family["preferred_age"] == "indifferent" or dog["age_group"] == family["preferred_age"]:
        reasons.append("fascia d'età compatibile")

    if family["preferred_size"] == "indifferent" or dog["maturity_size_label"] == family["preferred_size"]:
        reasons.append("taglia compatibile")

    if family["preferred_gender"] == "indifferent" or dog["gender_label"] == family["preferred_gender"]:
        reasons.append("sesso compatibile")

    if family["preferred_fur"] == "indifferent" or dog["fur_length_label"] == family["preferred_fur"]:
        reasons.append("lunghezza del pelo compatibile")

    breed_reason = explain_breed_match(dog, family)

    if breed_reason is not None:
        reasons.append(breed_reason)

    if dog["bert_similarity"] > all_dogs["bert_similarity"].quantile(0.75):
        reasons.append("descrizione semanticamente vicina al cane ideale")

    if len(reasons) == 0:
        return "Compatibilità basata sul punteggio complessivo."

    return ", ".join(reasons)


ranking_with_explanations = ranking.copy()

ranking_with_explanations["explanation"] = ranking_with_explanations.apply(
    lambda row: explain_recommendation(row, family_profile, dogs_experiment),
    axis=1
)

ranking_with_explanations[[
    "Name",
    "breed1_label",
    "breed2_label",
    "visual_breed_1",
    "visual_breed_1_score",
    "breed_score",
    "final_score",
    "explanation"
]]

,Name,breed1_label,breed2_label,visual_breed_1,visual_breed_1_score,breed_score,final_score,explanation
5743,Bobby,Golden Retriever,NaN,,0.000000,1.0,0.965444,"fascia d'età compatibile, taglia compatibile, ..."
8089,Pacino,Labrador Retriever,Mixed Breed,,0.000000,1.0,0.941526,"fascia d'età compatibile, sesso compatibile, l..."
7580,"Mummy Moon, Luna And Ariel",Golden Retriever,Labrador Retriever,,0.000000,1.0,0.938960,"fascia d'età compatibile, sesso compatibile, l..."
3697,2 Black Dude,Labrador Retriever,Mixed Breed,,0.000000,1.0,0.938751,"fascia d'età compatibile, sesso compatibile, l..."
2549,FLUFFY,Golden Retriever,NaN,,0.000000,1.0,0.937889,"fascia d'età compatibile, sesso compatibile, l..."
1285,Whiskey,Labrador Retriever,Mixed Breed,,0.000000,1.0,0.937493,"fascia d'età compatibile, sesso compatibile, l..."
1898,Nil,Labrador Retriever,Mixed Breed,,0.000000,1.0,0.927431,"fascia d'età compatibile, sesso compatibile, l..."
3309,Snow,Labrador Retriever,Mixed Breed,,0.000000,1.0,0.910484,"fascia d'età compatibile, taglia compatibile, ..."
37,Baby / Bubbles,Labrador Retriever,Jack Russell Terrier,,0.000000,1.0,0.904721,"taglia compatibile, sesso compatibile, lunghez..."
2757,Simba,Labrador Retriever,Beagle,,0.000000,1.0,0.887928,"taglia compatibile, sesso compatibile, lunghez..."


## 13. Conclusioni

Questo notebook integra le razze visive nel sistema di matching cane–famiglia.

La razza reale del dataset mantiene il peso maggiore, mentre la razza stimata dalle immagini ha peso minore e proporzionale alla confidenza del modello.

Questa estensione è utile soprattutto per i cani `Mixed Breed`, perché permette di riconoscere possibili somiglianze visive con le razze richieste dalla famiglia, senza trattarle come informazioni certe.
